# **RAG 구축하기_종합실습_나만의 챗봇 만들기**

* 예비 에이블러들을 위한 QA 챗봇 모델 만들기
    * Vector DB에 데이터 추가하기
    * Retriever, memory, LLM를 연결하기

## **1.환경준비**

### (1) 구글 드라이브

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(langchain)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

In [2]:
!pip install langchain langchain-community langchain-openai chromadb tiktoken pymupdf faiss-cpu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.8/449.8 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 94.4 MB/s eta 0:0

#### 2) 라이브러리 Import

In [3]:
import pandas as pd
import numpy as np
import os
import sqlite3
from datetime import datetime

import openai

from langchain_openai import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory, ConversationSummaryBufferMemory

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

### (3) OpenAI API Key 등록
* 환경변수로 key 등록

In [4]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/langchain/'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

In [5]:
print(os.environ['OPENAI_API_KEY'][:40])

sk-proj-9_V5Db5A_3EzpkFeDNheztR-ns32fKi-


## **2.Vector DB 만들기**

* 데이터 로딩
    * 제공한 csv 파일의 구조를 그대로 이용
    * 에이블스쿨 홈페이지 FAQ 데이터 수집(https://aivle.kt.co.kr/home/brd/faq/main?mcd=MC00000056)
        * 질문들을 csv 형태로 저장
    * csv 로더로 로딩하기

In [6]:
from langchain.document_loaders import CSVLoader

# CSV 파일 로드
csv_path = "aivleschool_qa_utf8.csv"
csv_loader = CSVLoader(file_path= path + csv_path)

# 문서 로드 실행
documents_csv = csv_loader.load()

# 첫 번째 행 출력
print(f"총 {len(documents_csv)} 개의 행이 로드됨")
print(documents_csv[0].page_content)
print(documents_csv[0].metadata)

총 10 개의 행이 로드됨
﻿구분: 모집/선발
QA: 최종 학력 또는 전공과 관계없이 지원할 수 있나요?
KT 에이블스쿨은 정규 4년제 대학 졸업자 및 졸업예정자를 대상으로 하는 교육입니다. 전공에 관계 없이 명시된 지원 자격에 부합한다면 모두 지원 가능합니다. 다만, AI개발자 Track은 기본적인 코딩역량이 필요합니다.
{'source': '/content/drive/MyDrive/langchain/aivleschool_qa_utf8.csv', 'row': 0}


* 벡터 데이터베이스
    * Embedding 모델 : text-embedding-ada-002
    * DB 경로 : ./db



In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
database = Chroma.from_documents(documents_csv, embeddings, persist_directory="./db")

* 입력된 데이터 조회

In [8]:
database.get()

{'ids': ['c9667f2c-eea2-4ac0-9ba7-784c55b3f3d7',
  'be2c7ddd-dce4-4cc2-a8ec-b4d85ce00718',
  '621aad9d-7ed9-4a32-8e81-aaf0ede76e72',
  '1a9e5d71-fcba-432f-a544-da1b5614f19b',
  '13703b90-7517-4e66-995c-495681a9ac17',
  '0798438f-5cfc-4a0e-b609-558233e63c23',
  '0cd36cbd-a127-4c7f-8f3c-f7e3ac6f86d4',
  '5898d401-a5fc-48ae-a3e2-f94616222a9e',
  'b8d39bc8-5459-464e-916e-8d4fa08aa3c2',
  'd85306b4-50aa-4146-97ca-7f74d0aeabd8'],
 'embeddings': None,
 'documents': ['\ufeff구분: 모집/선발\nQA: 최종 학력 또는 전공과 관계없이 지원할 수 있나요?\nKT 에이블스쿨은 정규 4년제 대학 졸업자 및 졸업예정자를 대상으로 하는 교육입니다. 전공에 관계 없이 명시된 지원 자격에 부합한다면 모두 지원 가능합니다. 다만, AI개발자 Track은 기본적인 코딩역량이 필요합니다.',
  '\ufeff구분: 모집/선발\nQA: 35세 이상은 지원할 수 없나요?\n본 교육 과정은 34세 이하를 대상으로 하는 교육입니다. 단, 모집시점에 35세라도 해당연도 1월 1일 이후 생일자는 지원이 가능합니다. (예: 5기 지원 기준 1989년 1월 1일 이후 출생자)',
  '\ufeff구분: 모집/선발\nQA: 미취업자의 기준이 뭔가요?\n미취업자의 기준은 아래와 같습니다.\n1) 기간의 정함이 있는 근로인 경우,\n2) 고용보험에 미가입한 경우,\n3) 고용보험에 가입되어 있더라도 15시간/주 미만 근로인 경우\n단, 어떠한 경우에도 교육을 풀타임(09:00~18:00)으로 들을 수 있어야 교육 참여가 가능합니다.',
  '

## **3.RAG 파이프라인**

* 모델 : ConversationalRetrievalChain
    * LLM 모델 : gpt-4.1-mini, temperature
    * retriever : 벡터DB
        * 유사도 높은 문서 3개 가져오도록 설정
    * 요약 memory
* ChatPromptTemplate로 역할 지정

In [9]:
# (1) 리트리버(Retriever) 생성
retriever = database.as_retriever(search_kwargs={"k": 3})

# (2) GPT-4.1-mini 모델 설정
llm = ChatOpenAI(model_name="gpt-4.1-mini")

# (3) 메모리 추가 (대화 문맥 유지)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

In [10]:
from langchain.prompts import ChatPromptTemplate

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 에이블스쿨 질문에 답변하는 챗봇이야. 답변은 간결하게 해줘."),
    ("human", "질문: {question}\n\n관련 문서:\n{context}")
])

# 체인 구성
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": prompt},
    return_source_documents=True,
)


* 모델 사용

In [11]:
# 질문
query = "지원하는데 나이 제한이 있나?"   # 질문할 문장

# 답변
result = qa_chain(query)
answer = result["answer"]
print(answer)

지원은 34세 이하를 대상으로 하며, 모집 시점에 35세라도 해당 연도 1월 1일 이후 생일자는 지원할 수 있습니다.
